## BERT 모델 학습 및 결과 테스트

In [13]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

transformers.logging.set_verbosity_error()


토크나이저 및 모델 로드

In [14]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def predict_sentiment(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    predicted_class = torch.argmax(outputs.logits, dim=-1).item()
    return "긍정 (Positive)" if predicted_class == 1 else "부정 (Negative)"


학습 전 모델 테스트

In [16]:
test_sentence = "I really wanted to like this movie, but it fell completely flat."

print("학습 전 모델 추론")
print(f"입력 문장: {test_sentence}")
print(f"예측 결과: {predict_sentiment(test_sentence)}") 


학습 전 모델 추론
입력 문장: I really wanted to like this movie, but it fell completely flat.
예측 결과: 긍정 (Positive)


데이터셋 로드

In [17]:
print("데이터셋 준비 중...")
dataset = load_dataset("imdb")
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(200))

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = small_train_dataset.map(tokenize_function, batched=True, desc="Train Tokenization")
tokenized_test = small_test_dataset.map(tokenize_function, batched=True, desc="Test Tokenization")



데이터셋 준비 중...


학습

In [18]:
print("\n모델 학습 시작...")
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="test_trainer",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    report_to="none" # 로그 깔끔하게 정리
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()
print("\n모델 학습 완료.\n")



모델 학습 시작...
{'eval_loss': 0.4606228172779083, 'eval_accuracy': 0.805, 'eval_runtime': 0.3271, 'eval_samples_per_second': 611.392, 'eval_steps_per_second': 39.74, 'epoch': 1.0}
{'eval_loss': 0.40611526370048523, 'eval_accuracy': 0.82, 'eval_runtime': 0.3546, 'eval_samples_per_second': 564.094, 'eval_steps_per_second': 36.666, 'epoch': 2.0}
{'eval_loss': 0.43730247020721436, 'eval_accuracy': 0.795, 'eval_runtime': 0.4122, 'eval_samples_per_second': 485.18, 'eval_steps_per_second': 31.537, 'epoch': 3.0}
{'train_runtime': 19.6429, 'train_samples_per_second': 152.727, 'train_steps_per_second': 9.622, 'train_loss': 0.3808220757378472, 'epoch': 3.0}

모델 학습 완료.



모델 추론

In [19]:
print("학습 후 모델 추론")
print(f"입력 문장: {test_sentence}")
print(f"예측 결과: {predict_sentiment(test_sentence)}")

학습 후 모델 추론
입력 문장: I really wanted to like this movie, but it fell completely flat.
예측 결과: 부정 (Negative)
